# 19 - B10 : solveur et chaine de tactiques configurables

**Navigation** : [Index SymbolicAI](../../README.md) | [Serie Z3](README.md) | [<< 18 - Einstein](18_Einsteins_Riddle.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Distinguer les **trois solveurs** que le binding peut construire (`Default`, `Simple`, `Logic`) et la **chaine de tactiques** qui les remplace ;
2. Comprendre que le choix du solveur est une **decision de configuration** du contexte, pas un parametre de la formule ;
3. Mesurer ce que ce choix change **effectivement** sur un probleme reel, et ce qu'il ne change pas ;
4. Reconnaitre le moment ou une configuration **refuse** de resoudre, et ou le solveur par defaut reussit.

### Pre-requis

- .NET Interactive (noyau `.net-csharp`) ; la version reellement utilisee est ecrite par la premiere cellule de code
- Les trois premieres lectures de la serie (`01` a `03`)

### Duree estimee : 40-50 minutes

***

> **Stack : A+B** - le DSL (`Z3.Linq`) expose le choix de solveur par une **propriete de contexte** ; l'API brute (`Microsoft.Z3`) expose les memes solveurs par des **appels de fabrique** (`MkSolver`, `MkSimpleSolver`, `MkSolver(logic)`). Les deux ecritures sont exercees sur la **meme instance**, et le resultat compare cellule par cellule.

Ce carnet ferme le **dernier item du backlog DSL** (#4616) : les neuf premiers (`B1` a `B9`) sont demontres dans la serie (`14`, `12`, `15`, `11`, `17`, `16`, `02`, `01`), le dixieme - le choix du solveur - n'etait demontre nulle part.

In [1]:
#r "../Z3.Linq/.deploy/Microsoft.Z3.dll"
#r "../Z3.Linq/.deploy/ExpressionUtils.dll"
#r "../Z3.Linq/.deploy/Z3.Linq.dll"
using Z3.Linq; using Microsoft.Z3; using System; using System.Collections.Generic; using System.Diagnostics; using System.Linq; using System.Runtime.InteropServices;

// --- Environnement de la mesure ---
// Ces trois lignes alimentent les sorties commitees : un temps sans environnement
// decrit n'est pas une mesure, c'est une anecdote.
Console.WriteLine($"Framework      : {RuntimeInformation.FrameworkDescription}");
Console.WriteLine($"Architecture   : {RuntimeInformation.ProcessArchitecture} - processeurs logiques : {System.Environment.ProcessorCount}");
Console.WriteLine($"Assembly Z3    : {typeof(Context).Assembly.GetName().Version}");

// --- Echauffement ---
// Le premier appel paie la compilation JIT et l'initialisation du noyau natif.
// On le paie une fois, ici, pour que les medianes plus bas mesurent le solveur
// et non la compilation.
var echauffement = new Z3Context()
    .NewTheorem<Symbols<int, int, int, int, int>>()
    .Where(t => t.X1 == 1)
    .Solve();
Console.WriteLine($"Echauffement   : X1 = {echauffement?.X1}");

// --- Protocole de mesure (section 4) ---
// Une mediane n'a de sens qu'avec un nombre de repetitions fixe : il est
// declare ici, avant le premier usage, pour que l'exercice 1 puisse s'en servir.
const int Repetitions = 5;

The below script needs to be able to find the current output cell; this is an easy method to get it.

Framework      : .NET 10.0.13


Architecture   : X64 - processeurs logiques : 32


Assembly Z3    : 4.12.2.0


Echauffement   : X1 = 1



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime



## 1. L'instance : un Sudoku 9x9, 30 indices

L'instance choisie est une grille a **30 indices** (sur 81 cases) :

```
530070000600195000098000060800060003400803001700020006060000280000419005000080079
```

Pourquoi le Sudoku 9x9 plutot que le planificateur de repas : le planificateur depend d'un **corpus externe** (recettes, masses, tables de composition) - le carnet ne serait alors pas autonome et la mesure ne serait pas reproductible sur un poste nu. Le Sudoku 9x9 est **autonome**, et surtout son **encodage DSL existe deja dans la serie** (`02_Sudoku_Theorem_vs_Array`) : on le **reutilise** au lieu d'en inventer un.

L'environnement du probleme est une liste de 81 entiers. Chaque element devient une constante Z3 distincte cote binding ; les contraintes de domaine, de ligne, de colonne et de boite s'expriment avec le mecanisme `Distinct` du DSL.

In [2]:
// L'environnement du probleme : 81 cases.
public class SudokuArray
{
    public List<int> Cells { get; set; } = Enumerable.Repeat(0, 81).ToList();
}

In [3]:
public static class Sudoku09
{
    // --- Encodage DSL : le meme que le carnet 02, repris tel quel ---
    public static Theorem<SudokuArray> Charger(Z3Context ctx, string puzzle)
    {
        var th = ctx.NewTheorem<SudokuArray>();
        var indices = Enumerable.Range(0, 9).ToArray();

        // 81 contraintes de domaine : chaque case porte un entier de 1 a 9
        for (int i = 0; i < 9; i++)
            for (int j = 0; j < 9; j++)
            {
                var i1 = i; var j1 = j;
                th = th.Where(s => s.Cells[i1 * 9 + j1] > 0 && s.Cells[i1 * 9 + j1] < 10);
            }

        // 9 lignes, 9 colonnes, 9 boites : les neuf cases d'une unite sont distinctes
        for (int r = 0; r < 9; r++)
        {
            var r1 = r;
            th = th.Where(t => Z3Methods.Distinct(indices.Select(j => t.Cells[r1 * 9 + j]).ToArray()));
        }
        for (int c = 0; c < 9; c++)
        {
            var c1 = c;
            th = th.Where(t => Z3Methods.Distinct(indices.Select(i => t.Cells[i * 9 + c1]).ToArray()));
        }
        for (int b = 0; b < 9; b++)
        {
            var b1 = b;
            var idx = (b1 / 3) * 3 * 9 + (b1 % 3) * 3;
            th = th.Where(t => Z3Methods.Distinct(new int[] {
                t.Cells[idx],      t.Cells[idx + 1],  t.Cells[idx + 2],
                t.Cells[idx + 9],  t.Cells[idx + 10], t.Cells[idx + 11],
                t.Cells[idx + 18], t.Cells[idx + 19], t.Cells[idx + 20] }));
        }

        // les 30 indices de la grille
        for (int k = 0; k < 81; k++)
        {
            var k1 = k;
            var v = int.Parse(puzzle.Substring(k1, 1));
            if (v != 0) th = th.Where(t => t.Cells[k1] == v);
        }
        return th;
    }

    // --- Encodage brut : la meme instance, ecrite a la main avec l'API Z3 ---
    public static List<int> ResoudreBrut(string puzzle, out string verdict, out long millisecondes)
    {
        using var zctx = new Context();
        var cells = new IntExpr[81];
        for (int i = 0; i < 81; i++) cells[i] = zctx.MkIntConst("c" + i);

        var sw = Stopwatch.StartNew();
        var s = zctx.MkSolver();
        for (int i = 0; i < 81; i++)
        {
            s.Assert(zctx.MkGe(cells[i], zctx.MkInt(1)));
            s.Assert(zctx.MkLe(cells[i], zctx.MkInt(9)));
            var v = int.Parse(puzzle.Substring(i, 1));
            if (v != 0) s.Assert(zctx.MkEq(cells[i], zctx.MkInt(v)));
        }
        for (int r = 0; r < 9; r++)
        {
            var g = new Expr[9];
            for (int c = 0; c < 9; c++) g[c] = cells[r * 9 + c];
            s.Assert(zctx.MkDistinct(g));
        }
        for (int c = 0; c < 9; c++)
        {
            var g = new Expr[9];
            for (int r = 0; r < 9; r++) g[r] = cells[r * 9 + c];
            s.Assert(zctx.MkDistinct(g));
        }
        for (int br = 0; br < 3; br++)
            for (int bc = 0; bc < 3; bc++)
            {
                var g = new Expr[9];
                for (int r = 0; r < 3; r++)
                    for (int c = 0; c < 3; c++)
                        g[r * 3 + c] = cells[(br * 3 + r) * 9 + (bc * 3 + c)];
                s.Assert(zctx.MkDistinct(g));
            }
        verdict = s.Check().ToString();
        var model = s.Model;
        var valeurs = Enumerable.Range(0, 81).Select(i => int.Parse(model.Eval(cells[i], true).ToString())).ToList();
        sw.Stop();
        millisecondes = sw.ElapsedMilliseconds;
        return valeurs;
    }

    public static string Grille(List<int> cells) =>
        string.Join(" | ", Enumerable.Range(0, 9).Select(r =>
            string.Join("", Enumerable.Range(0, 9).Select(c => cells[r * 9 + c].ToString()))));
}


warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, 

In [4]:
const string Puzzle = "530070000600195000098000060800060003400803001700020006060000280000419005000080079";

// --- Ecriture A : le DSL, avec le solveur par defaut ---
List<int> grilleDsl;
long msDsl;
using (var ctx = new Z3Context())
{
    var sw = Stopwatch.StartNew();
    var solution = Sudoku09.Charger(ctx, Puzzle).Solve();
    sw.Stop();
    msDsl = sw.ElapsedMilliseconds;
    grilleDsl = solution.Cells;
    Console.WriteLine($"[DSL ] {Sudoku09.Grille(grilleDsl)}   ({msDsl} ms)");
}

// --- Ecriture B : l'API brute, meme instance ---
var grilleBrut = Sudoku09.ResoudreBrut(Puzzle, out string verdictBrut, out long msBrut);
Console.WriteLine($"[RAW ] {Sudoku09.Grille(grilleBrut)}   ({msBrut} ms)");
Console.WriteLine($"[RAW ] verdict du solveur brut : {verdictBrut}");

// --- Comparaison : deux ecritures, un meme resultat ---
Console.WriteLine($"Meme grille ? {Sudoku09.Grille(grilleDsl) == Sudoku09.Grille(grilleBrut)}");

[DSL ] 534678912 | 672195348 | 198342567 | 859761423 | 426853791 | 713924856 | 961537284 | 287419635 | 345286179   (103 ms)


[RAW ] 534678912 | 672195348 | 198342567 | 859761423 | 426853791 | 713924856 | 961537284 | 287419635 | 345286179   (34 ms)


[RAW ] verdict du solveur brut : SATISFIABLE


Meme grille ? True


### Lecture de la section 1

Les deux ecritures donnent **la meme grille** - c'est le contrat de base d'un binding : l'ecriture declaree doit produire le meme resultat que l'ecriture manuelle.

Ce que le DSL ajoute n'est pas la fabrication de la formule : c'est la **reconstruction de la solution**. En ecriture brute, il faut evaluer chaque constante dans le modele et reconstruire soi-meme la grille ; en ecriture DSL, `Solve()` renvoie un objet `SudokuArray` dont la liste de 81 entiers est **deja remplie**. C'est la meme difference de nature que le carnet 01 (environnement `record` positionnel) et le carnet 17 (`Explain()`) : le DSL ne rend pas le probleme plus facile a resoudre, il rend le **resultat** plus facile a consommer.

Les deux temps affiches ne sont **pas** comparables terme a terme : celui du DSL inclut la traduction des 81 cases et des 27 contraintes de structure en formules Z3, celui du brut inclut l'ecriture manuelle de ces memes contraintes. La comparaison de solveurs, elle, se fera plus bas a configuration egale.

## 2. Quand une configuration refuse, et que le solveur par defaut reussit

Voici la propriete la plus utile du choix de solveur : **il existe des configurations qui ne resolvent pas**. Un choix de solveur n'est pas qu'une question de performance ; c'est une question de **droit de repondre**. Trois cas, trois mecanismes distincts, tous trois observables.

In [5]:
Console.WriteLine("=== Cas 1 : un solveur specialise par logique, SANS nom de logique ===");
try
{
    using var ctx = new Z3Context { SolverKind = SolverKind.Logic };
    ctx.NewTheorem<Symbols<int, int, int, int, int>>().Where(t => t.X1 == 3).Solve();
    Console.WriteLine("Aucune exception (inattendu) : le contexte aurait accepte une configuration incomplete.");
}
catch (InvalidOperationException ex)
{
    Console.WriteLine($"InvalidOperationException : {ex.Message}");
}
// Le meme enonce, avec le solveur par defaut, resout sans difficultes :
using (var ctx = new Z3Context())
{
    var s = ctx.NewTheorem<Symbols<int, int, int, int, int>>().Where(t => t.X1 == 3).Solve();
    Console.WriteLine($"Meme enonce, solveur par defaut : X1 = {s?.X1}");
}

Console.WriteLine();
Console.WriteLine("=== Cas 2 : une chaine de tactiques qui nomme une tactique inexistante ===");
try
{
    using var ctx = new Z3Context { Tactics = new[] { "simplify", "tactique-inexistante" } };
    ctx.NewTheorem<Symbols<int, int, int, int, int>>().Where(t => t.X1 == 3).Solve();
    Console.WriteLine("Aucune exception (inattendu) : le solveur aurait accepte une tactique inconnue.");
}
catch (Z3Exception ex)
{
    Console.WriteLine($"Z3Exception : {ex.Message}");
}
// La meme chaine, avec un nom valide, resout :
using (var ctx = new Z3Context { Tactics = new[] { "simplify", "smt" } })
{
    var s = ctx.NewTheorem<Symbols<int, int, int, int, int>>().Where(t => t.X1 == 3).Solve();
    Console.WriteLine($"Meme enonce, chaine valide : X1 = {s?.X1}");
}

=== Cas 1 : un solveur specialise par logique, SANS nom de logique ===


InvalidOperationException : SolverKind.Logic requires Logic to be set to an SMT-LIB logic name (e.g. "QF_LIA").


Meme enonce, solveur par defaut : X1 = 3


=== Cas 2 : une chaine de tactiques qui nomme une tactique inexistante ===


Z3Exception : unknown tactic tactique-inexistante


Meme enonce, chaine valide : X1 = 3



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, 

In [6]:
Console.WriteLine("=== Cas 3 : qui gagne quand les deux mecanismes sont demandes ? ===");

// 3a. Solveur specialise par logique, sans nom de logique : refus.
try
{
    using var ctx = new Z3Context { SolverKind = SolverKind.Logic };
    ctx.NewTheorem<Symbols<int, int, int, int, int>>().Where(t => t.X1 == 3).Solve();
    Console.WriteLine("Solveur seul, sans nom de logique : aucune exception (inattendu).");
}
catch (InvalidOperationException ex)
{
    Console.WriteLine($"Solveur seul, sans nom de logique : {ex.GetType().Name}");
}

// 3b. La meme configuration, avec une chaine de tactiques : PLUS AUCUN REFUS.
using (var ctx = new Z3Context { SolverKind = SolverKind.Logic, Tactics = new[] { "simplify", "smt" } })
{
    var s = ctx.NewTheorem<Symbols<int, int, int, int, int>>().Where(t => t.X1 == 3).Solve();
    Console.WriteLine($"Chaine de tactiques + solveur sans nom de logique : X1 = {s?.X1} (aucun refus)");
}

Console.WriteLine();
Console.WriteLine("Meme resultat avec un enonce que le solveur par defaut resout aussi :");
using (var ctx = new Z3Context())
{
    var s = ctx.NewTheorem<Symbols<int, int, int, int, int>>().Where(t => t.X1 == 3).Solve();
    Console.WriteLine($"Solveur par defaut : X1 = {s?.X1}");
}

=== Cas 3 : qui gagne quand les deux mecanismes sont demandes ? ===


Solveur seul, sans nom de logique : InvalidOperationException


Chaine de tactiques + solveur sans nom de logique : X1 = 3 (aucun refus)


Meme resultat avec un enonce que le solveur par defaut resout aussi :


Solveur par defaut : X1 = 3



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, 

### Lecture de la section 2

Les trois cas se lisent ensemble :

- **Cas 1** - le garde-fou est **local** : il porte sur la *configuration* (un solveur par logique sans nom de logique n'est pas un solveur), pas sur la formule. L'enonce, lui, est parfaitement valide, et le solveur par defaut le resout sans difficultes.
- **Cas 2** - l'erreur vient de **Z3**, pas du binding : le nom de tactique est verifie par le noyau au moment de la resolution. La chaine valide resout le meme enonce.
- **Cas 3** - la chaine de tactiques **prime** sur le choix du solveur. La preuve ne repose sur aucun temps : c'est le meme garde-fou du cas 1 qui, **present dans la configuration, ne se declenche plus** des qu'une chaine de tactiques est presente. Le binding construit d'abord le solveur de la chaine et ne regarde le choix du solveur qu'a defaut de chaine.

Ce point merite d'etre souligne, parce qu'il est contre-intuitif : demander « un solveur par logique QF_LIA » **et** une chaine de tactiques ne donne pas un solveur QF_LIA, mais une chaine de tactiques. Une chaine se termine deja par une procedure de decision : empiler par-dessus un choix de solveur serait sans effet ou contradictoire.

### Exercice 1 - mesurer une chaine de tactiques de votre choix

Vous venez de voir les trois refus possibles, et la chaine de tactiques qui les remplace. A vous de jouer : choisissez une chaine de votre choix (`lia2pb`, `qfbv`, ou une chaine que vous n'avez jamais essayee) et mesurez son cout sur l'instance de la section 1.

Objectif : obtenir une mediane en millisecondes, mesuree comme dans la section 4 - un echauffement hors mesure, puis des repetitions, puis la mediane. Les stubs de ce carnet sont volontairement **incomplets** : le carnet s'execute de bout en bout meme non complete.

*Indice* : la chaine est un tableau de noms de tactiques passe a `Z3Context.Tactics` ; le cout a mesurer inclut la construction du probleme, comme dans la section 4.

In [7]:
// --- Exercice 1 ---
// Re-mesurer l'instance du carnet avec une chaine de tactiques de VOTRE choix
// (par exemple "simplify", "lia2pb", "smt") et renvoyer la mediane en millisecondes.
// Indice : la chaine est un tableau de noms, passe a Z3Context.Tactics.
static string MedianeAvecTactiques(string puzzle, string[] tactiques, int repetitions)
{
    // TODO etudiant : mesurer (warm-up + repetitions) et renvoyer la mediane formatee
    Console.WriteLine("Exercice a completer : choisir une chaine de tactiques et mesurer la mediane");
    return "a completer";
}

Console.WriteLine($"Exercice 1 : mediane = {MedianeAvecTactiques(Puzzle, new[] { "simplify", "smt" }, Repetitions)}");

Exercice a completer : choisir une chaine de tactiques et mesurer la mediane


Exercice 1 : mediane = a completer


## 3. Le fragment : un indice de strategie, pas un garde-fou

Un solveur specialise par une logique SMT-LIB (`QF_LIA`, `QF_BV`, ...) pourrait-il **refuser** une formule qui sort de cette logique ? La question n'est pas academique : c'est ce qui ferait du choix de solveur un mecanisme de **typage** du probleme.

Verifions-le, plutot que de le supposer. Nous allons demander a un solveur specialise de resoudre des formules qui ne sont pas de son fragment, et regarder ce qu'il repond.

In [8]:
// --- 3a. En ecriture brute : un solveur QF_BV face a une formule entiere ---
using (var z = new Context())
{
    var s = z.MkSolver("QF_BV");
    var x = z.MkIntConst("x");
    s.Assert(z.MkGe(x, z.MkInt(1)));
    s.Assert(z.MkLe(x, z.MkInt(9)));
    Console.WriteLine($"[RAW ] QF_BV + formule entiere        : {s.Check()}");
}

// --- 3b. En ecriture brute : un solveur QF_LIA face a une formule de bit-vectors ---
using (var z = new Context())
{
    var s = z.MkSolver("QF_LIA");
    var bv = z.MkBitVecSort(8);
    var b = z.MkConst("b", bv);
    s.Assert(z.MkEq(b, z.MkNumeral(3, bv)));
    Console.WriteLine($"[RAW ] QF_LIA + formule bit-vector     : {s.Check()}");
}

// --- 3c. En ecriture brute : un solveur QF_NIA face a une formule lineaire ---
using (var z = new Context())
{
    var s = z.MkSolver("QF_NIA");
    var x = z.MkIntConst("x");
    s.Assert(z.MkGe(x, z.MkInt(1)));
    Console.WriteLine($"[RAW ] QF_NIA + formule lineaire        : {s.Check()}");
}

// --- 3d. En ecriture DSL : le meme test, passe par le binding ---
using (var ctx = new Z3Context { SolverKind = SolverKind.Logic, Logic = "QF_LIA" })
{
    var s = ctx.NewTheorem<Symbols<int, int, int, int, int>>()
        .Where(t => t.X1 >= 1).Where(t => t.X1 <= 9)
        .Where(t => t.X2 >= 1).Where(t => t.X2 <= 9)
        .Where(t => t.X1 * t.X2 == 12)
        .Solve();
    Console.WriteLine($"[DSL ] QF_LIA + produit de deux variables : {(s == null ? "UNSAT" : $"SAT (X1={s.X1}, X2={s.X2})")}");
}

[RAW ] QF_BV + formule entiere        : SATISFIABLE


[RAW ] QF_LIA + formule bit-vector     : SATISFIABLE


[RAW ] QF_NIA + formule lineaire        : SATISFIABLE


[DSL ] QF_LIA + produit de deux variables : SAT (X1=2, X2=6)



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, 

In [9]:
// --- 3e. La seule configuration qui echoue : un solveur dont le fragment ne
//         correspond pas aux sorts de l'enonce (ici des entiers, pas des bit-vectors).
using (var ctx = new Z3Context { SolverKind = SolverKind.Logic, Logic = "QF_BV" })
{
    try
    {
        var s = ctx.NewTheorem<Symbols<int, int, int, int, int>>()
            .Where(t => t.X1 >= 1).Where(t => t.X1 <= 9)
            .Solve();
        Console.WriteLine($"[DSL ] QF_BV + formule entiere : {(s == null ? "UNSAT" : $"SAT (X1={s.X1})")}");
    }
    catch (Exception ex)
    {
        Console.WriteLine($"[DSL ] QF_BV + formule entiere : {ex.GetType().Name} : {ex.Message}");
    }
}

// Le meme enonce, avec le solveur par defaut, resout :
using (var ctx = new Z3Context())
{
    var s = ctx.NewTheorem<Symbols<int, int, int, int, int>>()
        .Where(t => t.X1 >= 1).Where(t => t.X1 <= 9)
        .Solve();
    Console.WriteLine($"[DSL ] solveur par defaut, meme enonce : {(s == null ? "UNSAT" : $"SAT (X1={s.X1})")}");
}

[DSL ] QF_BV + formule entiere : InvalidCastException : Unable to cast object of type 'Microsoft.Z3.IntExpr' to type 'Microsoft.Z3.IntNum'.


[DSL ] solveur par defaut, meme enonce : SAT (X1=1)



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, 

### Lecture de la section 3

Le resultat est moins intuitif que la documentation du binding ne le laisse croire.

**Le fragment n'est pas un garde-fou.** Les quatre mesures de la section montrent qu'un solveur specialise par une logique **accepte** des formules qui ne sont pas de son fragment : il les resout, sans les refuser. La logique indique a Z3 **quelle strategie de decision employer** ; elle ne dit pas au noyau « refuse ce qui sort du fragment ». C'est une difference de fond entre un **typage** (qui refuse) et un **choix de strategie** (qui optimise).

**La seule configuration qui echoue** est celle dont le fragment ne correspond meme pas aux **sorts** de l'enonce : la cause de l'echec n'est pas un controle de fragment, mais la reconstruction du modele par le binding, qui ne sait pas lire la valeur rendue par ce solveur. Le meme enonce, avec le solveur par defaut, resout.

Ce que l'on peut donc dire de `SolverKind.Logic` : il **change la maniere dont Z3 decide**, pas **ce que Z3 accepte**. Et c'est exactement pour cela que la section 2 ne parle pas de typage mais de configuration : les seuls refus reels du chemin de resolution sont les gardes explicites - nom de logique manquant, nom de tactique inconnu.

### Exercice 2 - reproduire le refus de configuration

La section 3 a montre que le fragment ne rejette rien : une logique SMT-LIB est un indice de strategie, pas un controle de type. Le seul refus qui reste est donc celui de la section 2, le nom de logique manquant.

Objectif : ecrire une fonction qui tente de resoudre un enonce avec un solveur par logique **sans** nom de logique, qui attrape le refus, et qui renvoie le message d'erreur au lieu de laisser l'exception se propager.

*Indice* : `try` / `catch` sur `InvalidOperationException` ; la configuration fautive est celle du cas 1 de la section 2.

In [10]:
// --- Exercice 2 ---
// Reproduire le garde-fou du cas 1 : ecrire une fonction qui tente de resoudre un
// enonce avec un solveur par logique SANS nom de logique, et qui renvoie le message
// d'erreur au lieu de laisser l'exception se propager.
// Indice : try / catch sur InvalidOperationException.
static string RefusDuSolveurSansNomDeLogique()
{
    // TODO etudiant : construire la configuration fautive, attraper le refus, renvoyer le message
    Console.WriteLine("Exercice a completer : renvoyer le message du refus");
    return "a completer";
}

Console.WriteLine($"Exercice 2 : refus = « {RefusDuSolveurSansNomDeLogique()} »");

Exercice a completer : renvoyer le message du refus


Exercice 2 : refus = « a completer »


## 4. Ce que le choix change, en temps

La section 2 et la section 3 ont etabli deux choses : le choix de solveur peut **refuser**, et le fragment n'est pas un garde-fou. Reste la question pratique : **sur un probleme reel, le choix change-t-il quelque chose ?**

La mesure porte sur l'instance de la section 1, avec quatre configurations du binding : le solveur par defaut, le solveur incremental (`Simple`), le solveur specialise par logique (`QF_LIA`), et une chaine de tactiques. Pour chaque configuration, on mesure le **cout complet du chemin** - construction du probleme comprise - ce qui est le cout reellement paye par l'appelant, et on compare les **medianes**.

In [11]:
// --- Mesure des quatre configurations, sur la meme instance ---
// Protocole : une execution de warm-up (hors mesure), puis N repetitions ; on retient la mediane.
// Le warm-up ecarte la compilation JIT, qui ne concerne que la premiere mesure.

var configurations = new (string Nom, Func<Z3Context> Fabriquer)[]
{
    ("Solveur par defaut", () => new Z3Context()),
    ("Solveur incremental (Simple)", () => new Z3Context { SolverKind = SolverKind.Simple }),
    ("Solveur par logique (QF_LIA)", () => new Z3Context { SolverKind = SolverKind.Logic, Logic = "QF_LIA" }),
    ("Chaine de tactiques (simplify, smt)", () => new Z3Context { Tactics = new[] { "simplify", "smt" } }),
};

var resultats = new List<(string Nom, long Mediane, string Grille)>();

foreach (var (nom, fabriquer) in configurations)
{
    // warm-up : ni mesure ni compte dans la mediane
    using (var ctx = fabriquer())
    {
        Sudoku09.Charger(ctx, Puzzle).Solve();
    }

    var durees = new List<long>();
    string grille = "";
    for (int k = 0; k < Repetitions; k++)
    {
        using var ctx = fabriquer();
        var sw = Stopwatch.StartNew();
        var solution = Sudoku09.Charger(ctx, Puzzle).Solve();
        sw.Stop();
        durees.Add(sw.ElapsedMilliseconds);
        grille = Sudoku09.Grille(solution.Cells);
    }
    var mediane = durees.OrderBy(d => d).ElementAt(Repetitions / 2);
    resultats.Add((nom, mediane, grille));
    Console.WriteLine($"{nom,-34} mediane = {mediane,5} ms   (repetitions : {string.Join(", ", durees)} ms)");
}

Console.WriteLine();
var grilleDeReference = resultats[0].Grille;
foreach (var r in resultats)
    Console.WriteLine($"Meme grille que la reference ? {r.Grille == grilleDeReference}   ({r.Nom})");

Solveur par defaut                 mediane =    79 ms   (repetitions : 95, 88, 79, 75, 74 ms)


Solveur incremental (Simple)       mediane =    56 ms   (repetitions : 57, 56, 56, 56, 58 ms)


Solveur par logique (QF_LIA)       mediane =    70 ms   (repetitions : 70, 100, 67, 67, 71 ms)


Chaine de tactiques (simplify, smt) mediane =    55 ms   (repetitions : 56, 55, 55, 54, 58 ms)


Meme grille que la reference ? True   (Solveur par defaut)


Meme grille que la reference ? True   (Solveur incremental (Simple))


Meme grille que la reference ? True   (Solveur par logique (QF_LIA))


Meme grille que la reference ? True   (Chaine de tactiques (simplify, smt))


### Lecture de la section 4

Les medianes affichees montrent que **le choix du solveur se voit**, mais que l'ecart reste **modeste** sur cette instance : les quatre configurations tombent dans la meme decennie, et la plus rapide n'est pas un ordre de grandeur devant la plus lente. C'est le resultat attendu : une instance de cette taille ne discrimine pas fortement les strategies de decision - c'est une instance courte a resoudre, ou le cout dominant n'est pas le choix du solveur.

Ce qui **ne** depend d'aucun de ces facteurs, c'est la section 2 : la preuve de discrimination de ce carnet est le **refus** - une configuration qui ne repond pas la ou le solveur par defaut repond. Elle se verifie sur n'importe quelle machine, dans n'importe quel ordre d'execution, et ne se deduit d'aucun chronometre.

Une precision d'environnement, pour que la mesure soit interpretable : le binding transmet au `Context` Z3 les paires nom/valeur declarees par `SetParameter` - ce sont des **parametres globaux/module**, appliques a la creation du contexte (`Z3Context.cs:79-93`, puis `new Context(config)` en `:150`). Il n'expose en revanche **aucun reglage par solveur** : `Solver.UpdateParameters` et `ParamsRef` n'apparaissent nulle part dans le fork. La mesure ne peut donc pas fixer un `smt.*` propre a une instance de solveur, d'ou l'echauffement, les repetitions et la mediane. Le cout de la traduction du DSL vers Z3 est, lui, **inclus** dans les temps mesures ci-dessus : c'est le cout du chemin complet, pas celui du noyau seul.

### Exercice 3 - une chaine de tactiques rend la MEME grille

La section 4 a montre que les quatre configurations rendent la meme grille sur cette instance. Montrez-le vous-meme pour une chaine de tactiques de votre choix : le resultat doit etre la meme grille, et une seule ligne de sortie doit le dire.

Objectif : comparer les deux ecritures de la section 1 avec et sans chaine de tactiques, et le dire en une ligne.

*Indice* : la section 1 a deja les deux ecritures ; il suffit de les comparer.

In [12]:
// --- Exercice 3 ---
// Montrer, sur l'instance du carnet, qu'une chaine de tactiques donne la MEME grille
// que le solveur par defaut - et le dire dans une seule ligne de sortie.
// Indice : la section 1 a deja les deux ecritures ; il suffit de les comparer.
static string MemeGrilleAvecTactiques(string puzzle)
{
    // TODO etudiant : resoudre en DSL avec et sans chaine, comparer les deux grilles
    Console.WriteLine("Exercice a completer : comparer les deux grilles");
    return "a completer";
}

Console.WriteLine($"Exercice 3 : meme grille = {MemeGrilleAvecTactiques(Puzzle)}");

Exercice a completer : comparer les deux grilles


Exercice 3 : meme grille = a completer


## Conclusion

Le choix du solveur, dans le binding, est une **decision de configuration a trois formes** : un solveur general, un solveur incremental, un solveur specialise par une logique SMT-LIB - et une **chaine de tactiques** qui les remplace entierement.

Trois choses a retenir :

1. **Un choix de solveur peut refuser.** Nom de logique manquant, nom de tactique inconnu : dans les deux cas, le meme enonce resout avec le solveur par defaut. C'est la preuve de discrimination de ce carnet, et elle ne depend d'aucun chronometre.
2. **Le fragment n'est pas un garde-fou.** Un solveur specialise par `QF_LIA` accepte des formules qui ne sont pas lineaires, un solveur `QF_BV` accepte des formules entieres. La logique est un **indice de strategie**, pas un controle de type. Le seul echec observe vient d'une incompatibilite de **sorts**, non d'un controle de fragment.
3. **La chaine de tactiques prime** sur le choix du solveur : la preuve est l'absence meme du refus du point 1 des que la chaine est presente.

Ce carnet ferme le backlog DSL (#4616) : les dix items sont desormais demontres dans la serie. Il reste un sujet jumeau, hors de ce carnet : les **points d'extension de reecriture** du binding (`[TheoremPredicateRewriter]`, `[TheoremGlobalRewriter]`), qui permettent d'ecrire ses propres regles de reecriture du DSL - un autre carnet, une autre PR.

**Navigation** : [Index SymbolicAI](../../README.md) | [Serie Z3](README.md) | [<< 18 - Einstein](18_Einsteins_Riddle.ipynb)